In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import json
import os
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')


In [2]:
# Configuration stricte des chemins vers le nouveau répertoire
DOSSIER_RACINE = 'RN_sousdoss'
DOSSIER_MODELES = os.path.join(DOSSIER_RACINE, 'models')
DOSSIER_LOGS = os.path.join(DOSSIER_RACINE, 'logs')
DOSSIER_CLEANED = os.path.join(DOSSIER_RACINE, 'data_cleaned')
DOSSIER_TRANSFORMED = os.path.join(DOSSIER_RACINE, 'data_transformed')
DOSSIER_TF = os.path.join(DOSSIER_RACINE, 'data_tf_pipelines')

# Création de l'arborescence si elle n'existe pas
os.makedirs(DOSSIER_RACINE, exist_ok=True)
os.makedirs(DOSSIER_MODELES, exist_ok=True)
os.makedirs(DOSSIER_LOGS, exist_ok=True)
os.makedirs(DOSSIER_CLEANED, exist_ok=True)
os.makedirs(DOSSIER_TRANSFORMED, exist_ok=True)

In [3]:
df = pd.read_csv('Loan_default.csv', sep=';')

if 'LoanID' in df.columns:
    df = df.drop(columns=['LoanID'])

cible = 'Default'

df_temp, df_test = train_test_split(df, test_size=0.15, stratify=df[cible], random_state=42)
df_train, df_val = train_test_split(df_temp, test_size=0.1765, stratify=df_temp[cible], random_state=42)

colonnes_continues = df.select_dtypes(include=['float64', 'int64']).columns.drop(cible, errors='ignore').tolist()
colonnes_categorielles = df.select_dtypes(include=['object', 'category']).columns.tolist()

ensembles = {
    'train': df_train.copy(), 
    'val': df_val.copy(), 
    'test': df_test.copy()
}

print(f"Dimensions - Entrainement: {ensembles['train'].shape}, Validation: {ensembles['val'].shape}, Test: {ensembles['test'].shape}")

Dimensions - Entrainement: (178735, 17), Validation: (38309, 17), Test: (38303, 17)


In [4]:
valeurs_imputation = {'continues': {}, 'categorielles': {}}
dictionnaire_fillna = {}

for col in colonnes_continues:
    mediane = float(ensembles['train'][col].median())
    valeurs_imputation['continues'][col] = mediane
    dictionnaire_fillna[col] = mediane

for col in colonnes_categorielles:
    mode_val = str(ensembles['train'][col].mode().iloc[0])
    valeurs_imputation['categorielles'][col] = mode_val
    dictionnaire_fillna[col] = mode_val

for nom in ensembles.keys():
    ensembles[nom] = ensembles[nom].fillna(value=dictionnaire_fillna)

print("Valeurs manquantes restantes dans le jeu d'entrainement :", ensembles['train'].isnull().sum().sum())

Valeurs manquantes restantes dans le jeu d'entrainement : 0


In [5]:
limite_basse = 0.01
limite_haute = 0.99
limites_winsorisation = {}

for col in colonnes_continues:
    valeur_min = float(ensembles['train'][col].quantile(limite_basse))
    valeur_max = float(ensembles['train'][col].quantile(limite_haute))
    
    limites_winsorisation[col] = {'min': valeur_min, 'max': valeur_max}
    
    for nom in ensembles.keys():
        ensembles[nom][col] = ensembles[nom][col].clip(lower=valeur_min, upper=valeur_max)

with open(os.path.join(DOSSIER_RACINE, 'valeurs_imputation.json'), 'w') as f:
    json.dump(valeurs_imputation, f)

with open(os.path.join(DOSSIER_RACINE, 'limites_winsorisation.json'), 'w') as f:
    json.dump(limites_winsorisation, f)

print("Winsorisation terminee. Parametres sauvegardes au format JSON.")

Winsorisation terminee. Parametres sauvegardes au format JSON.


In [6]:
dossier_export = 'data_cleaned'
os.makedirs(os.path.join(DOSSIER_RACINE, dossier_export), exist_ok=True)

ensembles['train'].to_csv(os.path.join(DOSSIER_RACINE, dossier_export, 'train.csv'), index=False, sep=';')
ensembles['val'].to_csv(os.path.join(DOSSIER_RACINE, dossier_export, 'val.csv'), index=False, sep=';')
ensembles['test'].to_csv(os.path.join(DOSSIER_RACINE, dossier_export, 'test.csv'), index=False, sep=';')

print(f"Exportation des donnees terminee dans le repertoire : {dossier_export}")

Exportation des donnees terminee dans le repertoire : data_cleaned
